# Part 4 - Model Development
Using 5 fold cross validation on the training set.

# Part 4.1: Dummy models
Run dummy classifiers and a basic heuristic.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

In [2]:
import mlflow
from mlflowFunction import log_model_run

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment("Model Development")
mlflow.utils.logging_utils.disable_logging()


Tracking URI: http://127.0.0.1:5000


In [3]:

train_df = pd.read_parquet('C:\\Users\\Hzaab\\Desktop\\MLSD project\\data\\preprocessed\\train_processed.parquet', engine='fastparquet')
TARGET_COL = 'fake'

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL]

cv = KFold(n_splits=5, shuffle=True, random_state=10)

from sklearn.model_selection import cross_validate

def evaluate_model(model, X, y, cv):
    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "recall": "recall",
        "precision": "precision"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results = {
        "accuracy": scores["test_accuracy"].mean(),
        "f1": scores["test_f1"].mean(),
        "recall": scores["test_recall"].mean(),
        "precision": scores["test_precision"].mean()
    }

    print(
        f"{model.__class__.__name__} | "
        f"Acc: {results['accuracy']:.4f} | "
        f"F1: {results['f1']:.4f} | "
        f"Recall: {results['recall']:.4f} | "
        f"Precision: {results['precision']:.4f}"
    )

    return results

In [4]:
# Model that always outputs the majority class
print('--- Majority Class Dummy ---')
dummy_majority = DummyClassifier(strategy='most_frequent')
scores=evaluate_model(dummy_majority, X, y, cv)
log_model_run("Dummy Majority", dummy_majority, scores, X, y)

# Model that outputs randomly
print('\n--- Random Dummy ---')
dummy_random = DummyClassifier(strategy='uniform')
scores=evaluate_model(dummy_random, X, y, cv)
log_model_run("Dummy Random", dummy_random, scores, X, y)

# Model with a simple heuristic
print('\n--- Simple Heuristic (Decision Stump) ---')
# Using a decision tree with max_depth=1 as a robust heuristic for splitting on a single feature attribute
heuristic_clf = DecisionTreeClassifier(max_depth=1, random_state=10)
scores=evaluate_model(heuristic_clf, X, y, cv)
log_model_run("Decision Stump", heuristic_clf, scores, X, y)

--- Majority Class Dummy ---
DummyClassifier | Acc: 0.8000 | F1: 0.0000 | Recall: 0.0000 | Precision: 0.0000
🏃 View run Dummy Majority at: http://127.0.0.1:5000/#/experiments/2/runs/1cfe1b21927d45708a23e39c9c7961cf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- Random Dummy ---
DummyClassifier | Acc: 0.4690 | F1: 0.2612 | Recall: 0.4711 | Precision: 0.1810
🏃 View run Dummy Random at: http://127.0.0.1:5000/#/experiments/2/runs/47733668411a4661af1ec79e8d5f8299
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- Simple Heuristic (Decision Stump) ---
DecisionTreeClassifier | Acc: 0.9445 | F1: 0.8522 | Recall: 0.8006 | Precision: 0.9132
🏃 View run Decision Stump at: http://127.0.0.1:5000/#/experiments/2/runs/0167a8e0d6c84c70a4ad45f9efc979e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


## Part 4.2: Simple Models

In [5]:
# Logistic Regression
print('--- Logistic Regression ---')
log_reg = LogisticRegression(max_iter=1000, random_state=10)
scores=evaluate_model(log_reg, X, y, cv)
log_model_run("Logistic Regression", log_reg, scores, X, y)

# Decision Tree
print('\n--- Decision Tree ---')
tree_clf = DecisionTreeClassifier(random_state=10)
scores=evaluate_model(tree_clf, X, y, cv)
log_model_run("Decision Tree", tree_clf, scores, X, y)

# Random Forest
print('\n--- Random Forest ---')
rf_clf = RandomForestClassifier(n_estimators=100, random_state=10)
scores=evaluate_model(rf_clf, X, y, cv)
log_model_run("Random Forest", rf_clf, scores, X, y)

# K-Nearest Neighbors
print('\n--- K-Nearest Neighbors ---')
knn_clf = KNeighborsClassifier()
scores=evaluate_model(knn_clf, X, y, cv)
log_model_run("K-Nearest Neighbors", knn_clf, scores, X, y)


--- Logistic Regression ---
LogisticRegression | Acc: 0.9820 | F1: 0.9536 | Recall: 0.9297 | Precision: 0.9794
🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/a2da9ebdc9ba4148900719506a4ec778
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- Decision Tree ---
DecisionTreeClassifier | Acc: 0.9755 | F1: 0.9374 | Recall: 0.9291 | Precision: 0.9461
🏃 View run Decision Tree at: http://127.0.0.1:5000/#/experiments/2/runs/92a6c5d32fe24658b4e81aa9c6e9d718
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- Random Forest ---
RandomForestClassifier | Acc: 0.9850 | F1: 0.9609 | Recall: 0.9343 | Precision: 0.9892
🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/31ca52912b494b7fb289618fea110a70
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- K-Nearest Neighbors ---
KNeighborsClassifier | Acc: 0.9755 | F1: 0.9347 | Recall: 0.8844 | Precision: 0.9922
🏃 View run K-Nearest Neighbors at: http://127.0.0.1:50

## Part 4.2: ML Models

In [6]:
# Neural Network
print('--- Neural Network ---')
nn_clf = MLPClassifier(random_state=10, max_iter=500)
scores=evaluate_model(nn_clf, X, y, cv)
log_model_run("Neural Network", nn_clf, scores, X, y)

# XGBoost
print('\n--- XGBoost ---')
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=10)
scores=evaluate_model(xgb_clf, X, y, cv)
log_model_run("XGBoost", xgb_clf, scores, X, y)

# LightGBM
print('\n--- LightGBM ---')
lgbm_clf = LGBMClassifier(random_state=10)
scores=evaluate_model(lgbm_clf, X, y, cv)
log_model_run("LightGBM", lgbm_clf, scores, X, y)


--- Neural Network ---
MLPClassifier | Acc: 0.9850 | F1: 0.9608 | Recall: 0.9412 | Precision: 0.9819


c:\Users\Hzaab\Desktop\MLSD project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


🏃 View run Neural Network at: http://127.0.0.1:5000/#/experiments/2/runs/fe6524fb22514d4cbf399e1e3c01e173
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- XGBoost ---
XGBClassifier | Acc: 0.9830 | F1: 0.9562 | Recall: 0.9393 | Precision: 0.9739


c:\Users\Hzaab\Desktop\MLSD project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:21:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/f737d0b514aa440090af57c780828c6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

--- LightGBM ---
LGBMClassifier | Acc: 0.9860 | F1: 0.9638 | Recall: 0.9441 | Precision: 0.9849
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 400, number of negative: 1600
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1251
[LightGBM] [Info] Number of data points in the train set: 2000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [4]:
# svm
print('\n--- SVM ---')
svm_clf = SVC(random_state=10)
scores=evaluate_model(svm_clf, X, y, cv)
log_model_run("SVM", svm_clf, scores, X, y)


--- SVM ---
SVC | Acc: 0.9830 | F1: 0.9549 | Recall: 0.9168 | Precision: 0.9973
🏃 View run SVM at: http://127.0.0.1:5000/#/experiments/2/runs/ded1f4952aa04d50aac3a62e9c371e49
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
